# Ejercicio 01 — Agregar experiment tracking a un entrenamiento que ya funciona

El enunciado completo, con los criterios de completitud, está en
[`ejercicio-01.md`](ejercicio-01.md). Léelo antes de empezar.

## Qué hay aquí

Un entrenamiento que **ya funciona** y no registra nada. Tu tarea es agregarle
MLflow tracking en las celdas marcadas con `# TODO`, sin tocar el código base.

## Prerrequisitos

1. Las particiones materializadas: `make data` desde la raíz del repositorio.
2. El servidor de MLflow corriendo: `make mlflow` (queda en `127.0.0.1:5001`).
3. La UI abierta en <http://127.0.0.1:5001>.

---
## Parte 1 — Código base (no modificar, solo ejecutar)

Las celdas siguientes cargan los datos, entrenan el modelo y calculan las
métricas. Ejecútalas todas antes de continuar.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from taxi import config
from taxi.features import contract as fc
from taxi.models import train

# Los artefactos se escriben en un directorio temporal y de ahi se suben a MLflow.
# No se escriben en el repositorio: un artefacto que vive en tu disco y no en el
# artifact store no es trazable, y ensucia el `git status` de todo el mundo.
DIR_SALIDA = Path(tempfile.mkdtemp(prefix="ejercicio-01-"))
print("artefactos temporales en:", DIR_SALIDA)

In [ ]:
# --- Cargar datos ---
# Las particiones y las features vienen del paquete `taxi`: es la unica
# definicion del curso. Antes habia dos rutas de preprocesamiento distintas y
# este ejercicio leia pickles que ningun notebook generaba.
df_train = train.cargar_train()
df_valid = train.cargar_valid()

y_valid = df_valid[fc.TARGET_REGRESION].to_numpy(dtype=float)

print("train:", df_train.shape, [p.etiqueta for p in config.PARTICIONES_TRAIN])
print("valid:", df_valid.shape, config.PARTICION_VALID.etiqueta)
print("features:", fc.FEATURES)

In [ ]:
# --- Entrenar modelo ---
# 25 arboles para que la celda tarde alrededor de un minuto. Cronometrala.
n_estimators = 25
max_depth = 10
random_state = config.SEMILLA

modelo = train.pipeline_random_forest(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)
train.ajustar(modelo, df_train, df_valid)
y_pred = modelo.predict(df_valid)
print("Modelo entrenado.")

In [ ]:
# --- Calcular metricas ---
# root_mean_squared_error, no mean_squared_error(squared=False): ese parametro
# fue ELIMINADO de scikit-learn (verificado contra 1.9.0).
rmse = float(root_mean_squared_error(y_valid, y_pred))
mae = float(mean_absolute_error(y_valid, y_pred))
r2 = float(r2_score(y_valid, y_pred))

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

In [ ]:
# --- Generar tabla de predicciones ---
df_predicciones = pd.DataFrame(
    {"y_true": y_valid, "y_pred": y_pred, "error": y_valid - y_pred}
)
ruta_csv = DIR_SALIDA / "predictions.csv"
df_predicciones.to_csv(ruta_csv, index=False)
print(f"{len(df_predicciones)} filas en {ruta_csv.name}")
df_predicciones.head()

In [ ]:
# --- Generar grafica de residuales ---
fig, eje = plt.subplots(figsize=(8, 4))
eje.hist(y_valid - y_pred, bins=50, edgecolor="black", alpha=0.7)
eje.set_title("Distribucion de residuales (y_true - y_pred)")
eje.set_xlabel("residual (minutos)")
eje.set_ylabel("frecuencia")
eje.axvline(x=0, color="red", linestyle="--", alpha=0.6)
fig.tight_layout()

ruta_png = DIR_SALIDA / "residuals.png"
fig.savefig(ruta_png, dpi=150)
plt.show()
print("grafica en", ruta_png.name)

---

## Hasta aquí todo funciona, pero...

Los resultados están en la consola y en dos archivos de un directorio temporal.
Si mañana quieres comparar este entrenamiento con otro, no tienes con qué: no hay
registro de los parámetros, ni de los datos, ni del modelo.

**Tu tarea:** agregar MLflow tracking en las celdas siguientes.

---

## Parte 2 — Agregar tracking (completar los TODO)

### TODO 1: Configurar la conexion a MLflow

Importa `mlflow`, conectate al servidor y selecciona un experimento.

**Pistas:**
- Usa `mlflow.set_tracking_uri(...)` con la URL del servidor
- Usa `mlflow.set_experiment(...)` con un nombre descriptivo

In [ ]:
# TODO 1: Configurar la conexion a MLflow
# Pistas:
#   - mlflow.set_tracking_uri(...) con la URL del servidor. Puedes usar
#     config.MLFLOW_TRACKING_URI en lugar de escribir el puerto a mano.
#   - mlflow.set_experiment("nyc-taxi-ejercicio-01")
# Escribe tu codigo aqui:

### TODO 2-5: Registrar el run completo

Dentro de `with mlflow.start_run(...)` registra:

- **TODO 2 · tags** (`mlflow.set_tag(clave, valor)`), tres:
  - `problem_type` → `regression`
  - `model_family` → `random_forest`
  - `dataset` → las particiones de entrenamiento (usa
    `",".join(p.etiqueta for p in config.PARTICIONES_TRAIN)`; un tag de datos que
    no dice **qué** datos no sirve para nada)
- **TODO 3 · params** (`mlflow.log_param(nombre, valor)`), tres:
  `n_estimators`, `max_depth`, `random_state`
- **TODO 4 · métricas** (`mlflow.log_metric(nombre, valor)`), tres:
  `rmse`, `mae`, `r2`
- **TODO 5 · artefactos** (`mlflow.log_artifact(ruta)`), dos: el CSV y el PNG que
  generaste arriba (`ruta_csv` y `ruta_png`)

> Alternativa que conviene conocer: `mlflow.log_figure(fig, "graficos/x.png")` y
> `mlflow.log_table(df, "tablas/x.json")` escriben directo al artifact store, sin
> pasar por un archivo local. Aquí se usa `log_artifact` porque es la API que vas
> a encontrar en cualquier código existente.

In [ ]:
# TODO 2-5: Registrar el run completo
# Escribe tu codigo aqui:

with mlflow.start_run(run_name="rf-ejercicio-01"):
    # TODO 2: tags (3)

    # TODO 3: params (3)

    # TODO 4: metricas (3)

    # TODO 5: artefactos (2)

    print("Run registrado en MLflow.")

### Verificacion

Ejecuta la siguiente celda para confirmar que tu run se registro correctamente.
Deberia mostrar una tabla con tus metricas.

In [ ]:
# --- Verificacion: cuenta lo que pide la tabla de criterios ---
import mlflow  # deberia estar ya importado en el TODO 1

runs = mlflow.search_runs(
    experiment_names=["nyc-taxi-ejercicio-01"],
    order_by=["start_time DESC"],
)

if runs.empty:
    print("No se encontraron runs. Revisa el TODO 1 y el nombre del experimento.")
else:
    ultimo = runs.iloc[0]
    cliente = mlflow.MlflowClient()
    datos = cliente.get_run(ultimo["run_id"]).data
    artefactos = [a.path for a in cliente.list_artifacts(ultimo["run_id"])]
    propios = {k: v for k, v in datos.tags.items() if not k.startswith("mlflow.")}

    print(f"run_id      : {ultimo['run_id']}")
    print(f"tags        : {len(propios)} / 3  -> {sorted(propios)}")
    print(f"parametros  : {len(datos.params)} / 3  -> {sorted(datos.params)}")
    print(f"metricas    : {len(datos.metrics)} / 3  -> {sorted(datos.metrics)}")
    print(f"artefactos  : {len(artefactos)} / 2  -> {artefactos}")

---

## Paso final — verificar en la UI

1. Abre <http://127.0.0.1:5001>.
2. Busca el experimento **nyc-taxi-ejercicio-01**.
3. Confirma los 3 tags, 3 parámetros, 3 métricas y 2 artefactos.

La tabla de criterios de completitud está en
[`ejercicio-01.md`](ejercicio-01.md#criterios-de-completitud).

---

## Bonus (opcional)

1. Cambia `max_depth` a 20 y vuelve a ejecutar. Compara los dos runs en la UI y
   di **con qué evidencia** decides cuál es mejor.
2. Agrega un scatter `y_true` vs `y_pred` como artefacto, usando `log_figure`.
3. Sustituye tu logging manual por `mlflow.autolog()`. ¿Qué registra que tú no
   registraste? Y más importante: **¿qué NO registra?** (pista: mira tus tags).
4. Loguea el modelo con `signature` e `input_example`. Con este pipeline vas a
   necesitar `skops_trusted_types=["taxi.models.train.ADiccionarios"]`: descubre
   por qué leyendo el error que aparece si no lo pasas.